# Profileren van regionale netbelasting en storingen met PROC CHART

## Managementsamenvatting

Een netbeheeranalist gebruikt PROC CHART om een synthetische steekproef van metergegevens van voedingscircuits te profileren over vier bedieningsregio's en vier opwekbronnen. Het notebook doorloopt verticale staaf-, horizontale staaf-, blok- en cirkeldiagrammen om de opwekmix samen te vatten, de gemiddelde en totale belasting per regio te vergelijken, de avondpiek in de vraag per uur bloot te leggen en de storingsminuten per bron te rangschikken — het soort snelle, tekstuitvoergerichte verkenning dat een energie- en nutsteam uitvoert voordat het zich vastlegt op een grafisch rapport. De DATA-stap vraagt om 1.200 rijen; dit notebook is weergegeven in ongelicentieerde modus, die de werkende dataset beperkt tot de eerste 100 metingen, dus elk diagram hieronder vat die steekproef van 100 rijen samen.

## Gegevensbronnen

| Dataset | Rijen | Beschrijving |
|---------|------|-------------|
| `grid_ops` | 100 (synthetische steekproef) | Eén rij per metergegeven van een voedingscircuit op een regionaal net, inline gegenereerd met `call streaminit(20260531)` en `rand()`. De DATA-stap vraagt om 1.200 metingen, maar de ongelicentieerde modus beperkt de opgeslagen dataset tot 100 observaties, dus de diagrammen hieronder beschrijven die 100 rijen. |

# Profileren van regionale netbelasting en storingen met PROC CHART

PROC CHART produceert op tekens gebaseerde staaf-, blok- en cirkeldiagrammen rechtstreeks in de listing — er is geen ODS Graphics-apparaat nodig. Dat maakt het een snel verkenningsinstrument voor een eerste indruk voor een netbeheerteam dat de vorm van zijn belastings- en betrouwbaarheidsgegevens *wil zien* voordat het gepolijste GCHART- of SGPLOT-visualisaties bouwt.

In dit notebook doen we het volgende:

1. Een synthetische maand aan metergegevens van voedingscircuits genereren voor een regionale netbeheerder.
2. De **opwekmix** in kaart brengen (aandeel van metingen per bron).
3. De **gemiddelde en totale geleverde belasting** vergelijken over bedieningsregio's.
4. De **avondpiek in de vraag** per uur van de dag blootleggen.
5. Een **blokdiagram** gebruiken om regio tegen opwekbron te kruisen.
6. **Storingsminuten** per bron rangschikken om de minst betrouwbare voeding te vinden.

Elke instructie en optie hieronder is standaard SAS 9.4 PROC CHART-syntaxis.

## Stap 1 — De synthetische operationele gegevens genereren

De DATA-stap hieronder fabriceert metergegevens in een lus van 1.200 iteraties. Aan elke meting wordt een bedieningsregio, een opwekbron (Gas domineert, met Wind, Solar en Hydro voor de rest) en een uur van de dag toegewezen. De belasting is hoger in het avondvenster van 17:00–21:00 uur (en we markeren die metingen als piek), en we trekken storingsminuten uit een Poisson-verdeling. `call streaminit` legt de random seed vast zodat de gegevens reproduceerbaar zijn.

De NOTE in de log toont het praktische resultaat: deze run is in ongelicentieerde modus, dus de opgeslagen `grid_ops`-dataset is beperkt tot de eerste 100 van die metingen (100 rijen, 7 kolommen). Elk diagram dat volgt vat die steekproef van 100 rijen samen, en elke PROC CHART-log bevestigt dat er 100 rijen zijn gelezen.

In [1]:
/* Synthetic feeder-circuit operations for a regional grid operator */
GEGEVENS grid_ops;
    CALL streaminit(20260531);
    LENGTE region $12 source $9 peak_flag $3;
    REEKS regions[4] $12 _temporary_
        ('North','South','East','West');
    DOE meter_id = 1 TOT 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        ALS u < 0.42 DAN source = 'Gas';
        ANDERS ALS u < 0.70 DAN source = 'Wind';
        ANDERS ALS u < 0.88 DAN source = 'Solar';
        ANDERS source = 'Hydro';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = 'North') + 3 * (region = 'West');
        ALS hour >= 17 AND hour <= 21 DAN DOE;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'Yes';
        EINDE;
        ANDERS DOE;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = 'No';
        EINDE;
        ALS load_mwh < 0 DAN load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        UITVOER;
    EINDE;
    VERWIJDEREN r u BASE;
UITVOEREN;

NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds


## Stap 2 — Opwekmix

Welk aandeel van de metingen neemt elke opwekbron voor zijn rekening? Een verticaal staafdiagram met `TYPE=PERCENT` beantwoordt dit rechtstreeks: staafhoogten zijn het percentage van alle observaties dat in elke broncategorie valt. Omdat `source` een tekenvariabele is, behandelt PROC CHART de waarden ervan automatisch als discrete categorieën.

In [2]:
PROCEDURE chart GEGEVENS=grid_ops;
    VBAR source / type=PERCENT;
UITVOEREN;


Percent of SOURCE

         |              *****       
         |              *****       
   40.00 +              *****       
         |              *****       
         |              *****       
   30.00 +              *****       
         |       *****  *****       
         |       *****  *****       
   20.00 +       *****  *****       
         |       *****  *****       
         |*****  *****  *****       
         |*****  *****  *****  *****
   10.00 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          Hydro  Wind    Gas   Solar
                    SOURCE


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Stap 3 — Gemiddelde geleverde belasting per regio

Nu schakelen we over van tellen naar het samenvatten van een meting. Door `load_mwh` te benoemen in `SUMVAR=` met `TYPE=MEAN` wordt de staaflengte de gemiddelde belasting per regio, en we vragen de statistiekkolommen expliciet aan: `MEAN` drukt het gemiddelde naast elke staaf af en `CFREQ` voegt een kolom met cumulatieve frequentie toe. North draagt de hoogste gemiddelde geleverde belasting (gemiddelde ongeveer 28,0 MWh), consistent met de regionale offset die in de gegevens is ingebouwd, terwijl South het laagst is (ongeveer 19,8 MWh).

We geven ook `ASCENDING` mee, met de bedoeling de staven te ordenen van laagste naar hoogste gemiddelde. Merk in de uitvoer op dat de staven **niet** opnieuw geordend zijn — ze verschijnen in categorievolgorde (West, South, East, North), met gemiddelden 24,2, 19,8, 21,7, 28,0 die niet van kortste naar langste lopen. Het opnieuw ordenen van staven op basis van de diagramstatistiek is nog niet ingebouwd in deze PROC CHART-implementatie, dus `ASCENDING`/`DESCENDING` worden geaccepteerd maar hebben momenteel geen effect; lees de rangschikking in plaats daarvan af uit de afgedrukte `Mean`-kolom.

In [3]:
PROCEDURE chart GEGEVENS=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
UITVOEREN;


Mean of REGION

REGION                                                  Mean           N     Percent
                                                                                    
  West  **********************************             24.17       24.17       23.00
 South  ****************************                   19.80       43.97       21.00
  East  *******************************                21.72       65.69       29.00
 North  ****************************************       28.03       93.72       27.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Stap 4 — Totale belasting per regio

Hier maakt `TYPE=SUM` van elke staaf de *totale* geleverde belasting voor de regio in plaats van het gemiddelde, dus de hogere staven markeren de regio's die de meeste geaggregeerde energie leveren over de bemonsterde metingen. North leidt opnieuw op totale geleverde belasting.

De instructie vraagt ook `SUBGROUP=peak_flag` aan, waarbij PROC CHART wordt gevraagd elke staaf te splitsen op basis van de vraag of de meting in het avondpiekvenster viel. In SAS segmenteert dit elke staaf met een aparte glyph per subgroepniveau en drukt het een legenda af. Deze implementatie geeft subgroepsegmentatie nog niet weer (een gedocumenteerde beperking), dus de staven komen eruit als ononderbroken `*`-reeksen zonder uitsplitsing per segment — de staaf*totalen* zijn correct, maar de splitsing piek-versus-dal wordt hier niet getoond. Om te zien hoeveel belasting in piekuren valt, gebruikt u de uur-van-de-dagweergave in Stap 5.

In [4]:
PROCEDURE chart GEGEVENS=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
UITVOEREN;


Sum of REGION

         |                     *****
         |                     *****
         |                     *****
     600 +              *****  *****
         |*****         *****  *****
         |*****         *****  *****
         |*****         *****  *****
     400 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
     200 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          West   South  East   North
                    REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Stap 5 — Belastingsvorm gedurende de dag

`hour` is continu, dus we leggen de binning zelf vast met een expliciete `MIDPOINTS=`-lijst op middelpunten van 4 uur (2, 6, 10, 14, 18, 22). Elke staaf toont de gemiddelde geleverde belasting voor metingen nabij dat uur. De staaf gecentreerd op 18 zou moeten opvallen — dat is de avondpiek in de vraag die we in de gegevens hebben ingebouwd.

In [5]:
PROCEDURE chart GEGEVENS=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
UITVOEREN;


Mean of HOUR

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            HOUR


NOTE: PROC CHART data=grid_ops


## Stap 6 — Regio per opwekbron (blokdiagram)

Een BLOCK-diagram geeft een kleine tweewegtabel weer als een raster van blokken. Met `GROUP=source` en `SUMVAR=load_mwh / TYPE=MEAN` kruist het diagram elke regio tegen de opwekbron die deze bedient, met blokhoogte evenredig aan de gemiddelde belasting — een compacte manier om te zien welke combinaties van regio/bron de zwaarste gemiddelde belasting dragen.

In [6]:
PROCEDURE chart GEGEVENS=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
UITVOEREN;


Block chart of Mean of REGION by SOURCE

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West   South    East   North 
             REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Stap 7 — Opwekmix als cirkeldiagram

Dezelfde informatie over het bronaandeel uit Stap 2, getekend als een cirkel. PIE met `TYPE=PERCENT` bepaalt de grootte van elke sector op basis van het percentage van de totale metingen en drukt naast de figuur een legenda van sectorpercentages af.

In [7]:
PROCEDURE chart GEGEVENS=grid_ops;
    PIE source / type=PERCENT;
UITVOEREN;


Pie chart of SOURCE

          SOURCE     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
           Hydro       14.00     14.0%  ####
            Wind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Stap 8 — Storingsminuten per bron

Ten slotte rangschikken we de betrouwbaarheid. `SUMVAR=outage_min` met `TYPE=SUM` telt de storingsminuten per bron op. We geven `DESCENDING` mee om de slechtst presterende bron naar boven te laten drijven, maar net als in Stap 3 worden de staven niet opnieuw geordend — ze worden afgedrukt in categorievolgorde (Hydro, Wind, Gas, Solar) en het opnieuw ordenen van staven is nog niet geïmplementeerd. Lees de rangschikking af uit de afgedrukte `Sum`-kolom: Gas is goed voor de meeste totale storingsminuten in deze steekproef (122), ruim voor Wind (64), Solar (43) en Hydro (38). Dat volgt de opwekmix — Gas bedient de meeste metingen, dus het accumuleert in totaal de meeste storingsminuten.

In [8]:
PROCEDURE chart GEGEVENS=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum AFLOPEND;
UITVOEREN;


Sum of SOURCE

SOURCE                                                   Sum        Cum.     Percent
                                                                     Sum            
 Hydro  ************                                      38          38       14.00
  Wind  *********************                             64         102       28.00
   Gas  ****************************************         122         224       45.00
 Solar  **************                                    43         267       13.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## De resultaten interpreteren

De diagrammen samen gelezen geven het operationele team een snel situatiebeeld (over de steekproef van 100 rijen die deze run heeft vastgelegd):

- **Opwekmix (Stappen 2 en 7).** Gas draagt het grootste aandeel van de metingen (ongeveer 45%), met Wind op de tweede plaats (ongeveer 28%) en Hydro en Solar daarachter (ongeveer 14% en 13%) — de verticale staaf en de cirkel vertellen hetzelfde verhaal op twee manieren, een nuttige controle op consistentie.
- **Belasting per regio (Stappen 3 en 4).** De HBAR met gemiddelde belasting toont North met de hoogste gemiddelde geleverde belasting (gemiddelde ~28 MWh) en South de laagste (~20 MWh), consistent met de regionale offset die in de gegevens is ingebouwd. Het SUM-diagram bevestigt dat North ook de meeste totale energie levert.
- **Dagelijkse belastingsvorm (Stap 5).** De middelpuntstaaf gecentreerd op uur 18 is duidelijk de hoogste, wat de vraagpiek van 17:00–21:00 uur bevestigt die we in de gegevens hebben ingebouwd — precies waar een nutsbedrijf zich op vraagrespons en capaciteitsplanning zou richten.
- **Betrouwbaarheid (Stap 8).** Het optellen van storingsminuten per bron brengt Gas naar voren als de grootste bijdrager aan uitvaltijd in deze steekproef (122 minuten), het natuurlijke volgende doelwit voor onderhoudsevaluatie — hoewel dat vooral weerspiegelt dat Gas de meeste metingen bedient.

Twee weergaveopties die hier zijn gebruikt — het opnieuw ordenen van staven met `ASCENDING`/`DESCENDING` (Stappen 3 en 8) en de staafsegmentatie met `SUBGROUP=` (Stap 4) — worden door de parser geaccepteerd maar nog niet weergegeven door deze PROC CHART-implementatie, dus de rangschikkingen en splitsingen worden afgelezen uit de afgedrukte statistiekkolommen in plaats van uit de staafvolgorde of arcering.

PROC CHART levert alleen tekstuitvoer, dus voor bestuursklare visuals zou het team dezelfde weergaven verplaatsen naar PROC GCHART of PROC SGPLOT. Maar als eerste indruk zonder setup over een verse extractie beantwoorden deze diagrammen de operationele vragen — mix, omvang en timing — in seconden.